# Auslan → English: Uni-Sign official stage-3 recipe

This notebook is a new run, separate from `colab_train.ipynb` and its 3e-5/1e-4 experiments. It uses the official Uni-Sign pose-only downstream fine-tuning recipe, adapted to one A100:

- AdamW, peak learning rate `3e-4`, weight decay `1e-4`
- cosine decay, no warmup, 20 epochs
- micro-batch `8` × gradient accumulation `4` = effective batch `32`
- AdamW β=`(0.9, 0.999)`, ε=`1e-9`, gradient clipping `1.0`
- pose-only checkpoint, maximum pose length `256`, plain beam search with `num_beams=4`

The paper's reported loss is not a guaranteed target here: it used different datasets, language targets and much more pre-training. Also, with mT5's 250,112-token vocabulary and `label_smoothing=0.2`, this exact loss has a theoretical floor of about `2.99`; a loss of `2` is not attainable under the same objective. This run tests whether the optimization recipe was the bottleneck; compare Communication and News metrics separately. Run cells in order.

## 1. Runtime and dependencies

Use a GPU runtime. One A100 uses four accumulated micro-batches to match the paper's effective batch of 32 from four GPUs × batch 8.

In [12]:
import subprocess
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() or 'NO GPU')
DEVICE = 'cuda'

!pip -q install einops sacrebleu
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('CUDA is required for the real Uni-Sign run')

NVIDIA A100-SXM4-40GB, 40960 MiB
torch 2.11.0+cu128 | transformers 5.16.1 | cuda True


## 2. Mount Drive and locate the extracted poses

In [10]:
import os, glob, json, copy, hashlib, shutil, signal, subprocess, sys, tarfile
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive'
WORK = f'{DRIVE}/auslan_work'
MANIFEST = f'{WORK}/manifest.jsonl'
EXCLUDE = f'{WORK}/excluded.txt'
for path in (MANIFEST, EXCLUDE):
    if not os.path.exists(path):
        raise SystemExit(f'{path} is missing; run colab_setup.ipynb first')
print('work dir:', WORK)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
work dir: /content/drive/MyDrive/auslan_work


## 3. Copy the verified project code

In [5]:
CODE = '/content/unisign'

def locate(parts):
    for root in (DRIVE, '/content'):
        for prefix in ('', '*/', '*/*/'):
            hits = glob.glob(os.path.join(root, prefix, *parts))
            if hits:
                return hits[0]
    return None

src = locate(['unisign', 'spec.py'])
if src:
    src = os.path.dirname(src)
else:
    tarball = locate(['unisign_code.tar.gz'])
    if tarball is None:
        raise SystemExit('Upload unisign/ or unisign_code.tar.gz to Drive first')
    with tarfile.open(tarball) as tf:
        tf.extractall('/content/_code')
    src = '/content/_code/unisign'
if os.path.abspath(src) != CODE:
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(src, CODE)
sys.path.insert(0, CODE)
import spec
print('code from', src)
print('spec fingerprint', spec.SCHEMA_FINGERPRINT)
assert spec.SCHEMA_FINGERPRINT == 'bc3bb2df0f22948d', 'spec.py does not match extracted poses'
assert 'gradient_accumulation_steps' in open(f'{CODE}/train.py').read(), 'Upload the updated train.py / unisign_code.tar.gz first'

code from /content/drive/MyDrive/unisign
spec fingerprint bc3bb2df0f22948d


## 4. Download the pinned Uni-Sign model and mT5

This is the same verified CSL pose-only checkpoint used by the baseline.

In [6]:
from huggingface_hub import hf_hub_download, snapshot_download

INIT_CKPT = 'csl_stage1_weight.pth'
REPO_DIR = '/content/Uni-Sign'
REPO_COMMIT = 'eed438bcb49e30405cd6ccdfcccca330c134e830'
MT5_DIR = f'{REPO_DIR}/pretrained_weight/mt5-base'
MT5_REVISION = '2eb15465c5dd7f72a8f7984306ad05ebc3dd1e1f'
UNISIGN_REVISION = 'eab251b7fe7e8521afc0e67be98add670ea40a0d'
SHA256 = {
    'csl_stage1_weight.pth': '3c81cf4a087e9e81581e57a2f33f8f0acf87b518a1ada540a660a76cdb144ced',
    'mt5-base/pytorch_model.bin': '180573b534144580f04af026da62bf71bc976ee1b7eb311b8945e2fefde8d614',
}

if not os.path.isdir(f'{REPO_DIR}/.git'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ZechengLi19/Uni-Sign.git', REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-q', REPO_COMMIT], check=True)
snapshot_download('google/mt5-base', revision=MT5_REVISION, local_dir=MT5_DIR, allow_patterns=['*.json', '*.model', 'pytorch_model.bin'])
CKPT = hf_hub_download('ZechengLi19/Uni-Sign', INIT_CKPT, revision=UNISIGN_REVISION, local_dir='/content/checkpoints')

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            h.update(block)
    return h.hexdigest()

for name, path in [(INIT_CKPT, CKPT), ('mt5-base/pytorch_model.bin', f'{MT5_DIR}/pytorch_model.bin')]:
    if sha256(path) != SHA256[name]:
        raise SystemExit(f'{name}: sha256 mismatch')
    print('OK', name)
print('Uni-Sign code at', REPO_COMMIT[:7])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


OK csl_stage1_weight.pth
OK mt5-base/pytorch_model.bin
Uni-Sign code at eed438b


## 5. Unpack poses and write the official-aligned config

The run name is intentionally new. This is a separate BF16/no-warmup run; do not point it at the old FP32 `...__official_stage3__single_a100` folder or at the custom Arm A folders.

In [7]:
import yaml
POSE_LOCAL = '/content/pose'
os.makedirs(POSE_LOCAL, exist_ok=True)
rows = [json.loads(line) for line in open(MANIFEST)]
if len(glob.glob(f'{POSE_LOCAL}/*.npz')) < len(rows):
    chunks = sorted(glob.glob(f'{WORK}/pose/chunk_*.tar'))
    print(f'unpacking {len(chunks)} chunk(s) from Drive ...')
    for chunk in chunks:
        with tarfile.open(chunk) as tf:
            tf.extractall(POSE_LOCAL)
present = {os.path.basename(path)[:-4] for path in glob.glob(f'{POSE_LOCAL}/*.npz')}
missing = [row['uid'] for row in rows if row['uid'] not in present]
excluded = {line.strip() for line in open(EXCLUDE) if line.strip() and not line.startswith('#')}
print(f'{len(rows)} clips | {len(present)} poses | {len(missing)} missing | {len(excluded)} excluded')
if missing:
    raise SystemExit(f'{len(missing)} clips have no pose, e.g. {missing[:3]}')

# Official Uni-Sign Stage 3 is 4 GPUs × batch 8. This single-A100 run
# reproduces its effective batch with four accumulated micro-batches.
ARM = 'arm_a'
RUN = f'{ARM}__{INIT_CKPT.rsplit(".", 1)[0]}__official_stage3__single_a100__bf16'
BASE_CFG = yaml.safe_load(open(f'{CODE}/configs/{ARM}.yaml'))
CFG = copy.deepcopy(BASE_CFG)
CFG['seed'] = 0
CFG['num_workers'] = 8
CFG['log_every'] = 50
CFG['data'].update(manifest=MANIFEST, npz_dir=POSE_LOCAL, exclude=EXCLUDE, max_length=256)
CFG['backend'] = {'name': 'unisign', 'checkpoint': CKPT, 'repo': REPO_DIR, 'mt5_path': MT5_DIR, 'num_beams': 4, 'max_new_tokens': 100, 'label_smoothing': 0.2}
CFG['precision'] = 'bf16'
CFG['optim'].update(lr=3e-4, weight_decay=1e-4, batch_size=8, epochs=20, warmup_frac=0.0, grad_clip=1.0, trainable_groups=None, gradient_accumulation_steps=4, betas=[0.9, 0.999], eps=1e-9)
CFG['decode'] = {}
CFG['output_dir'] = f'{WORK}/runs/{RUN}'
CFG_PATH = f'{WORK}/train_configs/{RUN}.yaml'
os.makedirs(os.path.dirname(CFG_PATH), exist_ok=True)
with open(CFG_PATH, 'w') as fh:
    yaml.safe_dump(CFG, fh, sort_keys=False)
print(yaml.safe_dump({'run': RUN, 'data': CFG['data'], 'backend': CFG['backend'], 'optim': CFG['optim']}, sort_keys=False))
print('precision:', CFG['precision'])
print('effective batch:', CFG['optim']['batch_size'] * CFG['optim']['gradient_accumulation_steps'])

76549 clips | 76549 poses | 0 missing | 131 excluded
run: arm_a__csl_stage1_weight__official_stage3__single_a100__bf16
data:
  manifest: /content/drive/MyDrive/auslan_work/manifest.jsonl
  npz_dir: /content/pose
  max_length: 256
  gloss_text_mode: strip_paren
  exclude: /content/drive/MyDrive/auslan_work/excluded.txt
backend:
  name: unisign
  checkpoint: /content/checkpoints/csl_stage1_weight.pth
  repo: /content/Uni-Sign
  mt5_path: /content/Uni-Sign/pretrained_weight/mt5-base
  num_beams: 4
  max_new_tokens: 100
  label_smoothing: 0.2
optim:
  lr: 0.0003
  weight_decay: 0.0001
  batch_size: 8
  epochs: 20
  warmup_frac: 0.0
  grad_clip: 1.0
  trainable_groups: null
  gradient_accumulation_steps: 4
  betas:
  - 0.9
  - 0.999
  eps: 1.0e-09

precision: bf16
effective batch: 32


## 6. Train

`--resume` is safe on the first run and is required after a Colab disconnect. The updated `train.py` saves only at optimizer-step boundaries, so an interrupted accumulation group is replayed rather than silently discarded.

In [13]:
def run_train(extra, log_path=None):
    cmd = [sys.executable, '-u', 'train.py', '--config', CFG_PATH, '--device', DEVICE] + extra
    log = open(log_path, 'a') if log_path else None
    proc = subprocess.Popen(cmd, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end='', flush=True)
            if log:
                log.write(line); log.flush()
    except KeyboardInterrupt:
        print('Stop pressed: train.py will save a checkpoint ...', flush=True)
        proc.send_signal(signal.SIGINT)
        for line in proc.stdout:
            print(line, end='', flush=True)
            if log:
                log.write(line); log.flush()
    proc.wait()
    if log:
        log.close()
    return proc.returncode

OUT = CFG['output_dir']
METRICS = f'{OUT}/metrics.json'
if os.path.exists(METRICS):
    print('already finished:', OUT)
else:
    os.makedirs(OUT, exist_ok=True)
    rc = run_train(['--resume'], log_path=f'{OUT}/train.log')
    if rc == 130:
        raise SystemExit('Stopped safely; reconnect and re-run this cell to continue')
    if rc != 0:
        raise RuntimeError(f'train.py exited with code {rc}; checkpoint is retained')
print('metrics:', METRICS)

arm=arm_a device=cuda precision=bf16 out=/content/drive/MyDrive/auslan_work/runs/arm_a__csl_stage1_weight__official_stage3__single_a100__bf16
train=21998 val=1492
left out 119 clips listed in /content/drive/MyDrive/auslan_work/excluded.txt

Loading weights: 100%|██████████| 284/284 [00:00<00:00, 23214.50it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[load] missing=0 unexpected=0
trainable groups=['decoder', 'pose_encoder', 'temporal'] 587.75M / 587.75M (100.0%)
    decoder         582.40M  lr=3.00e-04
    pose_encoder      0.41M  lr=3.00e-04
    temporal          4.94M  lr=3.00e-04
batch=8 gradient_accumulation=4 effective_batch=32 updates/epoch=687
resumed from ckpt_step000013053.pt: step 13053/13740, epoch 19 batch 0
  {"step": 13100, "ep

## 7. Inspect loss and official plain-decoding metrics

The official comparison is the plain row (`num_beams=4`). `nr3` can be tested separately later, but it is a decoding ablation and must not replace the fixed official row.

In [14]:
subprocess.run([sys.executable, 'plot_loss.py', f'{OUT}/train_log.jsonl'], cwd=CODE, check=True)
with open(METRICS) as fh:
    metrics = json.load(fh)
print(json.dumps(metrics, indent=2, ensure_ascii=False))

{
  "auslandaily/communication": {
    "BLEU-1": 33.62,
    "BLEU-2": 17.71,
    "BLEU-3": 12.37,
    "BLEU-4": 9.77,
    "ROUGE-L": 20.55,
    "unique_hyps": 40.9,
    "looping": 2.4,
    "hyp_len": 5.1,
    "ref_len": 5.2,
    "n": 792,
    "oov": {
      "train_types": 13408,
      "test_types": 522,
      "oov_types": 4,
      "oov_type_rate": 0.007662835249042145,
      "oov_token_rate": 0.0009770395701025891,
      "examples": [
        "fluffy",
        "tasty",
        "catchup",
        "shines"
      ]
    }
  },
  "auslandaily/news": {
    "BLEU-1": 16.85,
    "BLEU-2": 5.57,
    "BLEU-3": 2.85,
    "BLEU-4": 1.82,
    "ROUGE-L": 12.33,
    "unique_hyps": 65.1,
    "looping": 37.9,
    "hyp_len": 18.4,
    "ref_len": 15.9,
    "n": 700,
    "oov": {
      "train_types": 13408,
      "test_types": 2831,
      "oov_types": 290,
      "oov_type_rate": 0.10243730130695868,
      "oov_token_rate": 0.02689618074233459,
      "examples": [
        "kambosos",
        "jr",
        

## Optional: diagnose repetition without changing the official score

This cell evaluates the same final weights with `no_repeat_ngram_size=3`. It writes to `eval_nr3/` and does not alter `metrics.json`.

In [15]:
NR3_DIR = f'{OUT}/eval_nr3'
if not os.path.exists(f'{NR3_DIR}/metrics.json'):
    rc = run_train(['--eval-only', '--eval-tag', 'nr3', '--set', 'decode.no_repeat_ngram_size=3'])
    if rc != 0:
        raise RuntimeError(f'nr3 evaluation failed with code {rc}')
else:
    print('already scored:', NR3_DIR)
if os.path.exists(f'{NR3_DIR}/metrics.json'):
    print(json.dumps(json.load(open(f'{NR3_DIR}/metrics.json')), indent=2, ensure_ascii=False))

arm=arm_a device=cuda precision=bf16 out=/content/drive/MyDrive/auslan_work/runs/arm_a__csl_stage1_weight__official_stage3__single_a100__bf16
train=21998 val=1492
left out 119 clips listed in /content/drive/MyDrive/auslan_work/excluded.txt

Loading weights: 100%|██████████| 284/284 [00:00<00:00, 22392.75it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[load] missing=0 unexpected=0
--eval-only: weights /content/drive/MyDrive/auslan_work/runs/arm_a__csl_stage1_weight__official_stage3__single_a100__bf16/checkpoint.pt (arm=arm_a step=13740), decode {'no_repeat_ngram_size': 3}
That's 100 lines that end in a tokenized period ('.')
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenize